In [1]:
%pip uninstall -y -q ipython-sql
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub jupysql

In [2]:
import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
%load_ext sql
%sql engine
%config SqlMagic.displaylimit = 50
%config SqlMagic.named_parameters = "disabled"

In [4]:
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [5]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [6]:
%%sql
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

++
||
++
++

In [7]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;


Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

99441 rows affected.

99441 rows affected.

32951 rows affected.

3095 rows affected.

112650 rows affected.

++
||
++
++

In [8]:
%%sql
SELECT 'customers' AS table_name,
       (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
       (SELECT COUNT(*) FROM olist_customers)     AS typed_rows
UNION ALL
SELECT 'orders',
       (SELECT COUNT(*) FROM olist_orders_raw),
       (SELECT COUNT(*) FROM olist_orders)
UNION ALL
SELECT 'order_items',
       (SELECT COUNT(*) FROM olist_order_items_raw),
       (SELECT COUNT(*) FROM olist_order_items)
UNION ALL
SELECT 'products',
       (SELECT COUNT(*) FROM olist_products_raw),
       (SELECT COUNT(*) FROM olist_products)
UNION ALL
SELECT 'sellers',
       (SELECT COUNT(*) FROM olist_sellers_raw),
       (SELECT COUNT(*) FROM olist_sellers)
ORDER BY table_name;


Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

5 rows affected.

table_name,raw_rows,typed_rows
customers,99441,99441
order_items,112650,112650
orders,99441,99441
products,32951,32951
sellers,3095,3095


Отже, кількість рядків у typed-таблицях збігається з raw-таблицями. Усі рядки пройшли приведення типів і constraints (PK, FK, CHECK) без втрат. Якби якийсь рядок порушив constraint, INSERT завершився б помилкою

Завдання 2. CTE-chain pipeline з ≥5 stages

In [9]:
%%sql
WITH
reference_date AS (
    -- Stage 0: стабільна reference date = наступний день після останнього замовлення
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + кількість позицій + прапорець скасування
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        (o.order_status = 'canceled') AS is_canceled,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COUNT(oi.order_item_id) AS items_cnt,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM per customer і monetary рахуємо лише по нескасованих замовленнях
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        COALESCE(SUM(s.order_total) FILTER (WHERE NOT s.is_canceled), 0)::NUMERIC(12, 2) AS monetary,
        SUM(s.items_cnt) AS total_items,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter + "життєвий цикл" клієнта в днях від першого замовлення
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter,
        (rd.as_of_date - r.first_order_date)::INTEGER AS customer_age_days
    FROM stage2_rfm AS r
    CROSS JOIN reference_date AS rd
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value лише по нескасованих замовленнях
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
    WHERE NOT s.is_canceled
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate = скасовані / всі замовлення клієнта
    SELECT
        customer_id,
        COUNT(*) FILTER (WHERE is_canceled) AS canceled_orders,
        (COUNT(*) FILTER (WHERE is_canceled)::NUMERIC / COUNT(*))::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: виручка по категоріях = тільки price (без доставки), без скасованих замовлень
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    WHERE o.order_status <> 'canceled'
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 категорія + її частка у витратах клієнта
    SELECT
        customer_id,
        product_category_name,
        category_revenue,
        category_share
    FROM (
        SELECT
            cr.*,
            (cr.category_revenue
                / NULLIF(SUM(cr.category_revenue) OVER (PARTITION BY cr.customer_id), 0)
            )::NUMERIC(6, 4) AS category_share,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
),
stage8_segment AS (
    -- Stage 8: бізнес-сегмент клієнта за recency + value tier (квартиль monetary)
    SELECT
        coh.customer_id,
        NTILE(4) OVER (ORDER BY coh.monetary) AS value_tier,
        CASE
            WHEN coh.monetary = 0          THEN 'no_revenue'
            WHEN coh.recency_days <= 90    THEN 'active'
            WHEN coh.recency_days <= 180   THEN 'cooling'
            ELSE                                'churn_risk'
        END AS segment
    FROM stage3_cohort AS coh
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    COALESCE(coh.total_items, 0) AS total_items,
    coh.first_order_date,
    coh.cohort_quarter,
    coh.customer_age_days,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_share, 0)::NUMERIC(6, 4) AS preferred_category_share,
    seg.value_tier,
    COALESCE(seg.segment, 'no_orders') AS segment
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
LEFT JOIN stage8_segment AS seg
    ON seg.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

customer_id,customer_state,recency_days,frequency,monetary,total_items,first_order_date,cohort_quarter,customer_age_days,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_share,value_tier,segment
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,8,2017-09-29,2017-07-01,384,13664.08,0.0000,telefonia_fixa,1.0000,4,churn_risk
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,4,2018-07-15,2018-07-01,95,7274.88,0.0000,telefonia_fixa,1.0000,4,cooling
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,1,2017-02-12,2017-01-01,613,6929.31,0.0000,utilidades_domesticas,1.0000,4,churn_risk
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,1,2018-07-25,2018-07-01,85,6922.21,0.0000,pcs,1.0000,4,active
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,1,2017-05-24,2017-04-01,512,6726.66,0.0000,artes,1.0000,4,churn_risk
05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,6,2017-11-24,2017-10-01,328,6081.54,0.0000,agro_industria_e_comercio,1.0000,4,churn_risk
df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,1,2017-04-01,2017-04-01,565,4950.34,0.0000,eletroportateis,1.0000,4,churn_risk
24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,1,2017-04-18,2017-04-01,548,4764.34,0.0000,eletroportateis,1.0000,4,churn_risk
3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,1,2018-06-22,2018-04-01,118,4681.78,0.0000,pcs,1.0000,4,cooling
1afc82cd60e303ef09b4ef9837c9505c,SP,76,1,4513.32,1,2018-08-03,2018-07-01,76,4513.32,0.0000,instrumentos_musicais,1.0000,4,active


Отже, pipeline складається з 8 named stages, кожен виконує одну трансформацію:
- stage1: сума замовлення;
- stage2: RFM;
- stage3: когорта;
- stage4: rolling AOV 30d (window function);
- stage5: cancellation rate;
- stage6: виручка по категоріях;
- stage7: top-1 категорія;
- stage8: сегмент.

Recency рахується від стабільної reference_date = MAX(order_purchase_timestamp) + 1, а не від CURRENT_DATE, тому результат відтворюваний.

Скасовані замовлення виключені з monetary, rolling AOV і виручки по категоріях, бо це не реальні гроші клієнта, і враховуються тільки в cancellation_rate. Виручка категорії рахується без freight_value. Додатково обчислено customer_age_days, preferred_category_share, value_tier і segment.

у топ-20 у всіх frequency = 1 і preferred_category_share = 1.0. Це особливість Olist: кожне замовлення отримує новий customer_id, а реальну людину ідентифікує customer_unique_id.


Завдання 3. Recursive CTE для referral chain

In [10]:
%%sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

-- Синтетичне бінарне дерево: клієнт з номером rn запрошує клієнтів 2*rn і 2*rn+1
WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 15
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id, referred_at)
SELECT
    parent.customer_id AS referrer_customer_id,
    child.customer_id  AS referred_customer_id,
    TIMESTAMPTZ '2018-01-01 00:00:00+00' + child.rn * INTERVAL '1 day' AS referred_at
FROM ordered_customers AS child
JOIN ordered_customers AS parent
    ON parent.rn = child.rn / 2
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT *
FROM hw5_referrals
ORDER BY referred_at;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

14 rows affected.

14 rows affected.

referrer_customer_id,referred_customer_id,referred_at
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2018-01-03 00:00:00+00:00
00012a2ce6f8dcda20d059ce98491703,0001fd6190edaaf884bcaf3d49edf079,2018-01-04 00:00:00+00:00
000161a058600d5901f007fab4c27140,0002414f95344307404f0ace7a26f1d5,2018-01-05 00:00:00+00:00
000161a058600d5901f007fab4c27140,000379cdec625522490c315e70c7a9fb,2018-01-06 00:00:00+00:00
0001fd6190edaaf884bcaf3d49edf079,0004164d20a9e969af783496f3408652,2018-01-07 00:00:00+00:00
0001fd6190edaaf884bcaf3d49edf079,000419c5494106c306a97b5635748086,2018-01-08 00:00:00+00:00
0002414f95344307404f0ace7a26f1d5,00046a560d407e99b969756e0b10f282,2018-01-09 00:00:00+00:00
0002414f95344307404f0ace7a26f1d5,00050bf6e01e69d5c0fd612f1bcfb69c,2018-01-10 00:00:00+00:00
000379cdec625522490c315e70c7a9fb,000598caf2ef4117407665ac33275130,2018-01-11 00:00:00+00:00
000379cdec625522490c315e70c7a9fb,0005aefbb696d34b3424dccd0a0e9fd0,2018-01-12 00:00:00+00:00


In [11]:
%%sql
WITH RECURSIVE referral_tree AS (
    -- ANCHOR: корені дерева = клієнти, яких ніхто не запросив (NOT EXISTS),
    --         але які самі когось запросили (EXISTS)
    SELECT
        c.customer_id,
        c.customer_id                AS root_customer_id,
        1                            AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT          AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )
      AND EXISTS (
        SELECT 1 FROM hw5_referrals AS r
        WHERE r.referrer_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: від referrer переходимо до його referred customers
    SELECT
        r.referred_customer_id,
        rt.root_customer_id,
        rt.depth + 1,
        rt.path_ids || r.referred_customer_id,
        rt.path_text || ' ---> ' || r.referred_customer_id
    FROM hw5_referrals AS r
    JOIN referral_tree AS rt
        ON rt.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rt.path_ids)   -- захист від циклів
      AND rt.depth < 50                                    -- depth guard
)
SELECT
    depth,
    COUNT(*)       AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_tree
GROUP BY depth
ORDER BY depth;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

4 rows affected.

depth,n_customers,sample_path
1,1,00012a2ce6f8dcda20d059ce98491703
2,2,00012a2ce6f8dcda20d059ce98491703 ---> 0001fd6190edaaf884bcaf3d49edf079
3,4,00012a2ce6f8dcda20d059ce98491703 ---> 0001fd6190edaaf884bcaf3d49edf079 ---> 000419c5494106c306a97b5635748086
4,8,00012a2ce6f8dcda20d059ce98491703 ---> 0001fd6190edaaf884bcaf3d49edf079 ---> 000419c5494106c306a97b5635748086 ---> 0009a69b72033b2d0ec8c69fc70ef768


Демонстрація захисту від циклу A → B → C → A

In [12]:
%%sql
WITH RECURSIVE
edges(referrer_id, referred_id) AS (
    -- штучний цикл A -> B -> C -> A
    VALUES ('A', 'B'), ('B', 'C'), ('C', 'A')
),
walk AS (
    -- ANCHOR
    SELECT 'A'::TEXT AS node, 1 AS depth, ARRAY['A']::TEXT[] AS path_ids
    UNION ALL
    -- RECURSIVE
    SELECT e.referred_id, w.depth + 1, w.path_ids || e.referred_id
    FROM edges AS e
    JOIN walk AS w ON w.node = e.referrer_id
    WHERE NOT e.referred_id = ANY(w.path_ids)   -- A вже є в path -> перехід C -> A відсікається
      AND w.depth < 50                           -- depth guard
)
SELECT depth, node, array_to_string(path_ids, ' -> ') AS path
FROM walk
ORDER BY depth;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

3 rows affected.

depth,node,path
1,A,A
2,B,A -> B
3,C,A -> B -> C


Синтетична таблиця hw5_referrals - це бінарне дерево з 15 клієнтів (14 пар): кожен клієнт rn запрошує 2·rn і 2·rn+1. А referred_at задано фіксованою датою, щоб результат був відтворюваним.

Recursive CTE:
- **Anchor** вибирає корені через NOT EXISTS. Це NULL-safe anti-join, на відміну від NOT IN
- **Recursive part** переходить referrer в referred
- В результаті на кожному рівні вдвічі більше клієнтів (1 - 2 - 4 - 8), максимальна глибина 4

Рекурсія завершується, коли recursive part не повертає нових рядків. Захист від циклів: NOT next_id = ANY(path_ids) і depth guard depth < 50. Демонстрація з циклом A → B → C → A зупиняється на 3-му рівні.

Завдання 4. Materialized View з CONCURRENT REFRESH

In [13]:
%%sql
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: стабільна reference date = наступний день після останнього замовлення
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + кількість позицій + прапорець скасування
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        (o.order_status = 'canceled') AS is_canceled,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COUNT(oi.order_item_id) AS items_cnt,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM per customer і monetary рахуємо лише по нескасованих замовленнях
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        COALESCE(SUM(s.order_total) FILTER (WHERE NOT s.is_canceled), 0)::NUMERIC(12, 2) AS monetary,
        SUM(s.items_cnt) AS total_items,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter + "життєвий цикл" клієнта в днях від першого замовлення
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter,
        (rd.as_of_date - r.first_order_date)::INTEGER AS customer_age_days
    FROM stage2_rfm AS r
    CROSS JOIN reference_date AS rd
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value лише по нескасованих замовленнях
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
    WHERE NOT s.is_canceled
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate = скасовані / всі замовлення клієнта
    SELECT
        customer_id,
        COUNT(*) FILTER (WHERE is_canceled) AS canceled_orders,
        (COUNT(*) FILTER (WHERE is_canceled)::NUMERIC / COUNT(*))::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: виручка по категоріях = тільки price (без доставки), без скасованих замовлень
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    WHERE o.order_status <> 'canceled'
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 категорія + її частка у витратах клієнта
    SELECT
        customer_id,
        product_category_name,
        category_revenue,
        category_share
    FROM (
        SELECT
            cr.*,
            (cr.category_revenue
                / NULLIF(SUM(cr.category_revenue) OVER (PARTITION BY cr.customer_id), 0)
            )::NUMERIC(6, 4) AS category_share,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
),
stage8_segment AS (
    -- Stage 8: бізнес-сегмент клієнта за recency + value tier (квартиль monetary)
    SELECT
        coh.customer_id,
        NTILE(4) OVER (ORDER BY coh.monetary) AS value_tier,
        CASE
            WHEN coh.monetary = 0          THEN 'no_revenue'
            WHEN coh.recency_days <= 90    THEN 'active'
            WHEN coh.recency_days <= 180   THEN 'cooling'
            ELSE                                'churn_risk'
        END AS segment
    FROM stage3_cohort AS coh
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    COALESCE(coh.total_items, 0) AS total_items,
    coh.first_order_date,
    coh.cohort_quarter,
    coh.customer_age_days,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_share, 0)::NUMERIC(6, 4) AS preferred_category_share,
    seg.value_tier,
    COALESCE(seg.segment, 'no_orders') AS segment
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
LEFT JOIN stage8_segment AS seg
    ON seg.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

99441 rows affected.

++
||
++
++

In [14]:
%%sql
-- n_rows має дорівнювати n_unique і кількості клієнтів (99 441):
-- одна строка на customer_id, без дублів
SELECT
    COUNT(*)                    AS n_rows,
    COUNT(DISTINCT customer_id) AS n_unique
FROM mv_olist_customer_features;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

1 rows affected.

n_rows,n_unique
99441,99441


In [15]:
%%sql
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

5 rows affected.

customer_id,customer_state,recency_days,frequency,monetary,total_items,first_order_date,cohort_quarter,customer_age_days,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_share,value_tier,segment
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,8,2017-09-29,2017-07-01,384,13664.08,0.0000,telefonia_fixa,1.0000,4,churn_risk
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,4,2018-07-15,2018-07-01,95,7274.88,0.0000,telefonia_fixa,1.0000,4,cooling
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,1,2017-02-12,2017-01-01,613,6929.31,0.0000,utilidades_domesticas,1.0000,4,churn_risk
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,1,2018-07-25,2018-07-01,85,6922.21,0.0000,pcs,1.0000,4,active
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,1,2017-05-24,2017-04-01,512,6726.66,0.0000,artes,1.0000,4,churn_risk


In [16]:
%%sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

++
||
++
++

MV mv_olist_customer_features містить повний 8-stage pipeline з Завдання 2.Перевірка n_rows = n_unique = 99 441 підтверджує, що в MV одна строчка на customer_id.

UNIQUE INDEX на customer_id обов'язковий для REFRESH ... CONCURRENTLY: PostgreSQL порівнює стару і нову версію MV по унікальному ключу і застосовує тільки різницю, тому читачі не блокуються під час оновлення. REFRESH MATERIALIZED VIEW CONCURRENTLY виконався без помилок.

Завдання 5. EXPLAIN ANALYZE + Reflection

In [17]:
%config SqlMagic.displaylimit = None

displaylimit: Value None will be treated as 0 (no limit)

In [18]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (-- Stage 0: стабільна reference date = наступний день після останнього замовлення
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + кількість позицій + прапорець скасування
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        (o.order_status = 'canceled') AS is_canceled,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COUNT(oi.order_item_id) AS items_cnt,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM per customer і monetary рахуємо лише по нескасованих замовленнях
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        COALESCE(SUM(s.order_total) FILTER (WHERE NOT s.is_canceled), 0)::NUMERIC(12, 2) AS monetary,
        SUM(s.items_cnt) AS total_items,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter + "життєвий цикл" клієнта в днях від першого замовлення
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter,
        (rd.as_of_date - r.first_order_date)::INTEGER AS customer_age_days
    FROM stage2_rfm AS r
    CROSS JOIN reference_date AS rd
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value лише по нескасованих замовленнях
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
    WHERE NOT s.is_canceled
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate = скасовані / всі замовлення клієнта
    SELECT
        customer_id,
        COUNT(*) FILTER (WHERE is_canceled) AS canceled_orders,
        (COUNT(*) FILTER (WHERE is_canceled)::NUMERIC / COUNT(*))::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: виручка по категоріях = тільки price (без доставки), без скасованих замовлень
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    WHERE o.order_status <> 'canceled'
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 категорія + її частка у витратах клієнта
    SELECT
        customer_id,
        product_category_name,
        category_revenue,
        category_share
    FROM (
        SELECT
            cr.*,
            (cr.category_revenue
                / NULLIF(SUM(cr.category_revenue) OVER (PARTITION BY cr.customer_id), 0)
            )::NUMERIC(6, 4) AS category_share,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
),
stage8_segment AS (
    -- Stage 8: бізнес-сегмент клієнта за recency + value tier (квартиль monetary)
    SELECT
        coh.customer_id,
        NTILE(4) OVER (ORDER BY coh.monetary) AS value_tier,
        CASE
            WHEN coh.monetary = 0          THEN 'no_revenue'
            WHEN coh.recency_days <= 90    THEN 'active'
            WHEN coh.recency_days <= 180   THEN 'cooling'
            ELSE                                'churn_risk'
        END AS segment
    FROM stage3_cohort AS coh
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    COALESCE(coh.total_items, 0) AS total_items,
    coh.first_order_date,
    coh.cohort_quarter,
    coh.customer_age_days,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_share, 0)::NUMERIC(6, 4) AS preferred_category_share,
    seg.value_tier,
    COALESCE(seg.segment, 'no_orders') AS segment
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
LEFT JOIN stage8_segment AS seg
    ON seg.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

179 rows affected.

QUERY PLAN
Limit (cost=91143.14..91143.19 rows=20 width=228) (actual time=3253.107..3257.643 rows=20 loops=1)
"Buffers: shared hit=12448, temp read=12714 written=12340"
CTE reference_date
-> Aggregate (cost=2915.40..2915.42 rows=1 width=4) (actual time=18.789..18.790 rows=1 loops=1)
Buffers: shared hit=1808
-> Seq Scan on olist_orders (cost=0.00..2693.92 rows=88592 width=8) (actual time=0.018..10.106 rows=99441 loops=1)
Buffers: shared hit=1808
CTE stage1_orders_total
-> Finalize GroupAggregate (cost=16949.66..28296.16 rows=88592 width=133) (actual time=182.126..471.772 rows=99441 loops=1)
Group Key: o_1.order_id


In [19]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

14 rows affected.

QUERY PLAN
Nested Loop (cost=0.83..8.98 rows=1 width=228) (actual time=0.043..0.044 rows=1 loops=1)
Buffers: shared hit=8
-> Limit (cost=0.42..0.55 rows=1 width=32) (actual time=0.019..0.020 rows=1 loops=1)
Buffers: shared hit=4
-> Index Only Scan using olist_customers_pkey on olist_customers (cost=0.42..10897.92 rows=84100 width=32) (actual time=0.018..0.018 rows=1 loops=1)
Heap Fetches: 1
Buffers: shared hit=4
-> Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv (cost=0.42..8.44 rows=1 width=228) (actual time=0.018..0.019 rows=1 loops=1)
Index Cond: (customer_id = olist_customers.customer_id)
Buffers: shared hit=4


In [20]:
%%sql
-- Індекс із шаблону: (monetary DESC) = (monetary DESC NULLS FIRST)
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

17 rows affected.

QUERY PLAN
Limit (cost=4705.01..4706.16 rows=10 width=228) (actual time=28.273..30.569 rows=10 loops=1)
Buffers: shared hit=1892
-> Gather Merge (cost=4705.01..11431.94 rows=58495 width=228) (actual time=28.271..30.565 rows=10 loops=1)
Workers Planned: 1
Workers Launched: 1
Buffers: shared hit=1892
-> Sort (cost=3705.00..3851.24 rows=58495 width=228) (actual time=25.416..25.418 rows=8 loops=2)
Sort Key: monetary DESC NULLS LAST
Sort Method: top-N heapsort Memory: 28kB
Buffers: shared hit=1892


In [21]:
%%sql
-- Індекс, що точно збігається з ORDER BY запиту
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary_nl
    ON mv_olist_customer_features (monetary DESC NULLS LAST);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

8 rows affected.

QUERY PLAN
Limit (cost=0.42..1.43 rows=10 width=228) (actual time=0.027..0.042 rows=10 loops=1)
Buffers: shared hit=10 read=3
-> Index Scan using idx_mv_olist_customer_features_monetary_nl on mv_olist_customer_features (cost=0.42..10020.03 rows=99441 width=228) (actual time=0.026..0.040 rows=10 loops=1)
Buffers: shared hit=10 read=3
Planning:
Buffers: shared hit=15 read=1
Planning Time: 0.196 ms
Execution Time: 0.057 ms


Коли я запустила EXPLAIN ANALYZE на повному pipeline, запит виконувався 3268 ms. А коли я брала дані одного клієнта з Materialized View, це зайняло всього 0.072 ms. Це приблизно в 45 000 разів швидше. У плані видно, що JOIN-и працюють швидко, Parallel Hash Join займає десь 100 ms. Найдовше виконуються сортування і групування. Наприклад, HashAggregate для RFM має Batches: 21 і пише 7720 kB на диск, бо даним не вистачає пам'яті. Також багато разів є Sort з external merge, тобто сортування теж іде через диск. Найдорожча ознака - це preferred_category. Для неї спочатку групуються дані по клієнту і категорії, а потім рахується ROW_NUMBER, і WindowAgg тут займає приблизно 380–860 ms.

Я думаю, що кешувати ознаки через MV варто. Пошук клієнта в MV йде через Index Scan по унікальному індексу і читає лише 8 буферів, тому це дуже швидко. Оскільки дані в нас змінюються раз на день, я б оновлювала MV один раз на добу вночі через REFRESH CONCURRENTLY, а також вручну, якщо завантажили нові замовлення.

Звичайний VIEW краще використати тоді, коли потрібні завжди свіжі дані і запит простий. Наприклад, коли треба подивитися замовлення одного клієнта.

Recursive CTE зупиняється, коли наступний крок не знаходить нових рядків. Якщо є цикл A → B → A, то без захисту запит би ніколи не закінчився. Тому я перевіряю, щоб клієнта ще не було в path_ids, і ставлю обмеження depth < 50. У моєму прикладі з циклом A → B → C → A запит зупинився на третьому рівні.

Для цього pipeline найважливіше, мені здається, баланс між свіжістю даних і вартістю перерахунку. Рахувати все заново на кожен запит занадто довго, а дані з затримкою в один день нас влаштовують. Ще я помітила, що індекс monetary DESC не спрацював через NULLS LAST, а з індексом monetary DESC NULLS LAST запит прискорився з 30.6 ms до 0.057 ms.